# Deepfake Audio Detection — Wav2Vec2 Training

This notebook trains a **Wav2Vec2-based deepfake audio detector** that can detect modern AI-generated speech.

**Architecture:** `facebook/wav2vec2-base` (pre-trained) + attentive statistics pooling + MLP classifier

**Instructions:**
1. Set runtime to **T4 GPU** (Runtime → Change runtime type)
2. Run all cells in order
3. Download `best_wav2vec_model.pth` when training completes


In [ ]:
!pip install -q transformers torchaudio kagglehub scikit-learn soundfile


In [ ]:
import kagglehub
import os

print("Downloading dataset...")
dataset_path = kagglehub.dataset_download("mohammedabdeldayem/the-fake-or-real-dataset")
print("Dataset path:", dataset_path)

# Find training directory
train_dirs = []
for root, dirs, files in os.walk(dataset_path):
    basename = os.path.basename(root).lower()
    if basename in ("training", "train"):
        train_dirs.append(root)

TRAIN_DIR = train_dirs[0] if train_dirs else dataset_path
print("Training directory:", TRAIN_DIR)

# Count files
for subdir in os.listdir(TRAIN_DIR):
    full = os.path.join(TRAIN_DIR, subdir)
    if os.path.isdir(full):
        count = len([f for f in os.listdir(full) if f.endswith((".wav", ".flac"))])
        print(f"  {subdir}: {count} files")


In [ ]:
"""
Wav2Vec2-Based Model for Deepfake Audio Detection.

Uses facebook/wav2vec2-base as a pre-trained feature extractor with a
lightweight classification head. This architecture captures phase coherence,
prosody, and micro-timing features that Mel-spectrograms discard â€” enabling
detection of modern TTS engines (ElevenLabs, TTSMP3, Bark, etc.).

Reference: This approach mirrors winning solutions from ASVspoof 2021/2024.
"""

import torch
import torch.nn as nn
from transformers import Wav2Vec2Model, Wav2Vec2Config


class Wav2Vec2DeepfakeDetector(nn.Module):
    """
    Wav2Vec2-based binary classifier for deepfake audio detection.
    
    Architecture:
        1. Wav2Vec2-Base feature extractor (pre-trained, partially frozen)
        2. Weighted layer aggregation across all transformer layers
        3. Attentive statistics pooling over time
        4. Lightweight MLP classification head
    
    Accepts raw 16kHz waveforms. No spectrogram preprocessing needed.
    """

    WAV2VEC2_MODEL_NAME = "facebook/wav2vec2-base"
    HIDDEN_SIZE = 768  # Wav2Vec2-Base hidden dimension

    def __init__(self, num_frozen_layers: int = 8) -> None:
        """
        Args:
            num_frozen_layers: Number of bottom transformer layers to freeze.
                Wav2Vec2-Base has 12 transformer layers. Setting this to 8
                freezes the bottom 8 and fine-tunes the top 4.
        """
        super().__init__()
        
        # Load pre-trained Wav2Vec2
        self.wav2vec2 = Wav2Vec2Model.from_pretrained(self.WAV2VEC2_MODEL_NAME)
        
        # Freeze the CNN feature encoder entirely
        self.wav2vec2.feature_extractor._freeze_parameters()
        
        # Freeze the bottom N transformer layers
        for i, layer in enumerate(self.wav2vec2.encoder.layers):
            if i < num_frozen_layers:
                for param in layer.parameters():
                    param.requires_grad = False
        
        # Learnable weighted sum across all 13 hidden states (1 CNN + 12 transformer)
        self.layer_weights = nn.Parameter(torch.ones(13) / 13)
        
        # Attentive statistics pooling â€” learns which time frames matter
        self.attention = nn.Sequential(
            nn.Linear(self.HIDDEN_SIZE, 128),
            nn.Tanh(),
            nn.Linear(128, 1)
        )
        
        # Classification head
        # Input: mean + std from attentive pooling = 2 * HIDDEN_SIZE
        self.classifier = nn.Sequential(
            nn.Linear(self.HIDDEN_SIZE * 2, 256),
            nn.BatchNorm1d(256),
            nn.ReLU(),
            nn.Dropout(0.3),
            nn.Linear(256, 1)
        )

    def forward(self, waveform: torch.Tensor) -> torch.Tensor:
        """
        Forward pass.
        
        Args:
            waveform: Raw audio tensor of shape (batch_size, num_samples).
                      Expected 16kHz, mono, float32 in range [-1, 1].
        
        Returns:
            Logits tensor of shape (batch_size,) for binary classification.
        """
        # Extract all hidden states from Wav2Vec2
        outputs = self.wav2vec2(waveform, output_hidden_states=True)
        hidden_states = outputs.hidden_states  # tuple of 13 tensors, each (B, T, 768)
        
        # Weighted sum across layers
        stacked = torch.stack(hidden_states, dim=0)  # (13, B, T, 768)
        weights = torch.softmax(self.layer_weights, dim=0)
        weighted = (stacked * weights.view(-1, 1, 1, 1)).sum(dim=0)  # (B, T, 768)
        
        # Attentive statistics pooling
        attn_scores = self.attention(weighted)  # (B, T, 1)
        attn_weights = torch.softmax(attn_scores, dim=1)  # (B, T, 1)
        
        mean = (weighted * attn_weights).sum(dim=1)  # (B, 768)
        std = torch.sqrt(
            ((weighted - mean.unsqueeze(1)) ** 2 * attn_weights).sum(dim=1) + 1e-8
        )  # (B, 768)
        
        pooled = torch.cat([mean, std], dim=1)  # (B, 1536)
        
        # Classify
        logits = self.classifier(pooled)
        return logits.squeeze(-1)  # (B,)

    def get_trainable_params_count(self) -> dict:
        """Returns count of trainable vs frozen parameters."""
        trainable = sum(p.numel() for p in self.parameters() if p.requires_grad)
        frozen = sum(p.numel() for p in self.parameters() if not p.requires_grad)
        return {"trainable": trainable, "frozen": frozen, "total": trainable + frozen}


In [ ]:
"""
Raw Waveform Data Pipeline for Wav2Vec2-Based Deepfake Detection.

Loads audio as raw 16kHz waveforms (no spectrogram conversion) and applies
RawBoost augmentation during training. RawBoost simulates real-world audio
degradation (codec artifacts, noise, filtering) to improve generalization
against modern TTS engines.

Reference: RawBoost â€” H. Tak et al., "RawBoost: A Raw Data Boosting and 
Augmentation Method applied to Automatic Speaker Verification Anti-Spoofing" (2022)
"""

import os
import logging
import random
from pathlib import Path
from typing import Tuple, List, Optional, Union, Dict

import numpy as np
import torch
import soundfile as sf
from torch.utils.data import Dataset, DataLoader
from torchaudio import transforms as T

# Configure logging
logging.basicConfig(
    level=logging.INFO,
    format='%(asctime)s - %(name)s - %(levelname)s - %(message)s'
)
logger = logging.getLogger(__name__)


class RawBoostAugmentor:
    """
    Implements RawBoost-inspired audio augmentation on raw waveforms.
    
    Applies a random combination of:
    1. Additive colored noise (pink/brown noise, not just white)
    2. Codec simulation (quantization + dithering)
    3. Random filtering (simple IIR bandpass-like effect)
    4. Random gain perturbation
    """

    def __init__(self, sample_rate: int = 16000) -> None:
        self.sample_rate = sample_rate

    def __call__(self, waveform: torch.Tensor) -> torch.Tensor:
        """
        Apply random augmentation to a waveform tensor of shape (num_samples,).
        """
        x = waveform.numpy()
        
        # Randomly apply 1-3 augmentations
        augmentations = [
            self._additive_noise,
            self._codec_simulation,
            self._random_filter,
            self._gain_perturbation,
        ]
        num_augs = random.randint(1, 3)
        chosen = random.sample(augmentations, num_augs)
        
        for aug in chosen:
            x = aug(x)
        
        return torch.from_numpy(x).float()

    def _additive_noise(self, x: np.ndarray) -> np.ndarray:
        """Add colored noise (pink noise via cumulative sum of white noise)."""
        snr_db = random.uniform(15, 40)  # Conservative SNR range
        noise = np.random.randn(len(x))
        
        # Make it pink noise (1/f spectrum) by cumulative filtering
        if random.random() > 0.5:
            noise = np.cumsum(noise)
            noise = noise - np.mean(noise)
        
        # Scale noise to desired SNR
        signal_power = np.mean(x ** 2) + 1e-10
        noise_power = np.mean(noise ** 2) + 1e-10
        scale = np.sqrt(signal_power / (noise_power * (10 ** (snr_db / 10))))
        
        return (x + scale * noise).astype(np.float32)

    def _codec_simulation(self, x: np.ndarray) -> np.ndarray:
        """Simulate low-bitrate codec by quantization + dithering."""
        bits = random.choice([8, 12, 14])  # Reduce bit depth
        max_val = 2 ** (bits - 1)
        
        # Quantize
        dither = np.random.uniform(-0.5 / max_val, 0.5 / max_val, len(x))
        quantized = np.round(x * max_val + dither) / max_val
        
        return quantized.astype(np.float32)

    def _random_filter(self, x: np.ndarray) -> np.ndarray:
        """Apply a simple random IIR filter to perturb frequency response."""
        # Single-pole IIR: y[n] = x[n] + coeff * y[n-1]
        coeff = random.uniform(-0.95, 0.95)
        y = np.zeros_like(x)
        y[0] = x[0]
        for i in range(1, len(x)):
            y[i] = x[i] + coeff * y[i - 1]
        
        # Normalize to prevent clipping
        max_abs = np.max(np.abs(y)) + 1e-10
        if max_abs > 1.0:
            y = y / max_abs
        
        return y.astype(np.float32)

    def _gain_perturbation(self, x: np.ndarray) -> np.ndarray:
        """Apply random gain change."""
        gain_db = random.uniform(-6, 6)
        gain = 10 ** (gain_db / 20)
        result = x * gain
        
        # Clip to prevent overflow
        return np.clip(result, -1.0, 1.0).astype(np.float32)


class RawWaveformDataset(Dataset):
    """
    PyTorch Dataset that loads raw waveforms for Wav2Vec2 inference.
    
    Unlike the Mel-spectrogram dataset, this returns raw float32 waveforms
    at 16kHz, padded/truncated to a fixed duration.
    """

    def __init__(
        self,
        data_dir: Union[str, Path],
        metadata_file: Optional[Union[str, Path]] = None,
        target_sample_rate: int = 16000,
        target_duration: float = 4.0,
        class_mapping: Optional[Dict[str, int]] = None,
        is_training: bool = False
    ) -> None:
        """
        Args:
            data_dir: Path to directory containing audio files.
            metadata_file: Optional protocol file with labels (ASVspoof format).
            target_sample_rate: Sample rate to resample all audio to.
            target_duration: Duration in seconds to pad/truncate to.
            class_mapping: Mapping from string labels to integer classes.
            is_training: If True, apply RawBoost augmentation.
        """
        self.data_dir = Path(data_dir)
        self.metadata_file = Path(metadata_file) if metadata_file else None
        self.target_sample_rate = target_sample_rate
        self.target_length = int(target_sample_rate * target_duration)
        self.is_training = is_training
        
        self.class_mapping = class_mapping or {
            "genuine": 0, "bonafide": 0, "real": 0,
            "spoof": 1, "deepfake": 1, "fake": 1
        }
        
        self.augmentor = RawBoostAugmentor(sample_rate=target_sample_rate) if is_training else None
        self.resampler_cache = {}  # Cache resamplers for different source rates
        
        self.file_paths: List[Path] = []
        self.labels: List[int] = []
        
        self._load_dataset_info()

    def _load_dataset_info(self) -> None:
        """Parse directory structure or metadata file to find audio files and labels."""
        if not self.data_dir.exists():
            raise FileNotFoundError(f"Data directory {self.data_dir} does not exist.")

        if self.metadata_file and self.metadata_file.exists():
            logger.info(f"Loading metadata from {self.metadata_file}")
            with open(self.metadata_file, 'r') as f:
                for line in f:
                    parts = line.strip().split()
                    if len(parts) >= 5:
                        filename = parts[1]
                        label_str = parts[-1].lower()
                        
                        file_path = self.data_dir / f"{filename}.flac"
                        if not file_path.exists():
                            file_path = self.data_dir / f"{filename}.wav"
                        
                        if file_path.exists() and label_str in self.class_mapping:
                            self.file_paths.append(file_path)
                            self.labels.append(self.class_mapping[label_str])
        else:
            logger.info("Inferring labels from directory structure.")
            for ext in ('*.wav', '*.flac'):
                for file_path in self.data_dir.rglob(ext):
                    parent_dir = file_path.parent.name.lower()
                    if parent_dir in self.class_mapping:
                        self.file_paths.append(file_path)
                        self.labels.append(self.class_mapping[parent_dir])
                    else:
                        filename = file_path.name.lower()
                        for key, val in self.class_mapping.items():
                            if key in filename:
                                self.file_paths.append(file_path)
                                self.labels.append(val)
                                break

        logger.info(f"Loaded {len(self.file_paths)} audio files. "
                     f"Real: {self.labels.count(0)}, Fake: {self.labels.count(1)}")

    def _load_and_preprocess(self, file_path: Path) -> torch.Tensor:
        """Load audio file, convert to mono 16kHz, pad/truncate."""
        audio_data, sample_rate = sf.read(str(file_path))
        
        # Convert to tensor
        if len(audio_data.shape) == 1:
            waveform = torch.from_numpy(audio_data).float()
        else:
            # Multi-channel â†’ mono by averaging
            waveform = torch.from_numpy(audio_data.mean(axis=1)).float()
        
        # Resample if needed
        if sample_rate != self.target_sample_rate:
            cache_key = sample_rate
            if cache_key not in self.resampler_cache:
                self.resampler_cache[cache_key] = T.Resample(
                    orig_freq=sample_rate, new_freq=self.target_sample_rate
                )
            waveform = self.resampler_cache[cache_key](waveform)
        
        # Pad or truncate to target length
        if waveform.shape[0] < self.target_length:
            pad_amount = self.target_length - waveform.shape[0]
            waveform = torch.nn.functional.pad(waveform, (0, pad_amount))
        elif waveform.shape[0] > self.target_length:
            # Random crop during training, center crop during eval
            if self.is_training:
                start = random.randint(0, waveform.shape[0] - self.target_length)
            else:
                start = (waveform.shape[0] - self.target_length) // 2
            waveform = waveform[start:start + self.target_length]
        
        # Normalize to [-1, 1]
        max_abs = waveform.abs().max()
        if max_abs > 0:
            waveform = waveform / max_abs
        
        return waveform

    def __len__(self) -> int:
        return len(self.file_paths)

    def __getitem__(self, idx: int) -> Tuple[torch.Tensor, int]:
        """Returns (waveform, label) where waveform is shape (target_length,)."""
        file_path = self.file_paths[idx]
        label = self.labels[idx]
        
        try:
            waveform = self._load_and_preprocess(file_path)
            
            # Apply RawBoost augmentation during training
            if self.is_training and self.augmentor is not None:
                waveform = self.augmentor(waveform)
            
            return waveform, label
            
        except Exception as e:
            logger.error(f"Failed to load {file_path}: {e}")
            return torch.zeros(self.target_length), label


In [ ]:
import torch
import torch.nn as nn
import numpy as np
from sklearn.metrics import accuracy_score, roc_curve
from torch.utils.data import random_split, DataLoader
from torch.cuda.amp import autocast, GradScaler
import time

# --- Configuration ---
BATCH_SIZE = 16
EPOCHS = 15
LR = 1e-5
WEIGHT_DECAY = 0.01
WARMUP_RATIO = 0.1
PATIENCE = 3

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Device: {device}")

# --- Load Dataset ---
print("Loading dataset...")
full_dataset = RawWaveformDataset(TRAIN_DIR, is_training=True)

# 80/20 split
train_size = int(0.8 * len(full_dataset))
val_size = len(full_dataset) - train_size
train_dataset, val_dataset = random_split(full_dataset, [train_size, val_size])
print(f"Train: {train_size}, Val: {val_size}")

train_loader = DataLoader(train_dataset, batch_size=BATCH_SIZE, shuffle=True, num_workers=2, pin_memory=True, drop_last=True)
val_loader = DataLoader(val_dataset, batch_size=BATCH_SIZE, shuffle=False, num_workers=2, pin_memory=True)

# --- Initialize Model ---
print("Loading Wav2Vec2 model...")
model = Wav2Vec2DeepfakeDetector(num_frozen_layers=8).to(device)
param_info = model.get_trainable_params_count()
print(f"Trainable: {param_info['trainable']:,} | Frozen: {param_info['frozen']:,} | Total: {param_info['total']:,}")

# --- Optimizer with layer-wise LR ---
classifier_params = list(model.classifier.parameters()) + list(model.attention.parameters()) + [model.layer_weights]
encoder_params = [p for p in model.wav2vec2.parameters() if p.requires_grad]

optimizer = torch.optim.AdamW([
    {"params": classifier_params, "lr": LR * 10},  # Higher LR for new layers
    {"params": encoder_params, "lr": LR},
], weight_decay=WEIGHT_DECAY)

criterion = nn.BCEWithLogitsLoss()
scaler = GradScaler()  # Mixed precision

# --- LR Scheduler with warmup ---
total_steps = len(train_loader) * EPOCHS
warmup_steps = int(total_steps * WARMUP_RATIO)

def lr_lambda(step):
    if step < warmup_steps:
        return step / max(1, warmup_steps)
    progress = (step - warmup_steps) / max(1, total_steps - warmup_steps)
    return 0.5 * (1 + np.cos(np.pi * progress))

scheduler = torch.optim.lr_scheduler.LambdaLR(optimizer, lr_lambda)

def compute_eer(y_true, y_scores):
    fpr, tpr, _ = roc_curve(y_true, y_scores)
    fnr = 1 - tpr
    idx = np.nanargmin(np.absolute(fnr - fpr))
    return fpr[idx]

# --- Training Loop ---
best_eer = float("inf")
epochs_no_improve = 0

for epoch in range(EPOCHS):
    start_time = time.time()
    
    # ---- TRAIN ----
    model.train()
    full_dataset.is_training = True
    train_loss = 0
    train_batches = 0
    
    for batch_idx, (waveforms, labels) in enumerate(train_loader):
        waveforms = waveforms.to(device)
        labels = labels.float().to(device)
        
        optimizer.zero_grad()
        
        with autocast():
            logits = model(waveforms)
            loss = criterion(logits, labels)
        
        scaler.scale(loss).backward()
        scaler.unscale_(optimizer)
        torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)
        scaler.step(optimizer)
        scaler.update()
        scheduler.step()
        
        train_loss += loss.item()
        train_batches += 1
        
        if (batch_idx + 1) % 50 == 0:
            print(f"  Batch {batch_idx+1}/{len(train_loader)} | Loss: {loss.item():.4f}")
    
    avg_train_loss = train_loss / max(1, train_batches)
    
    # ---- VALIDATE ----
    model.eval()
    full_dataset.is_training = False
    y_true, y_scores, y_pred = [], [], []
    
    with torch.no_grad():
        for waveforms, labels in val_loader:
            waveforms = waveforms.to(device)
            logits = model(waveforms)
            probs = torch.sigmoid(logits)
            
            y_true.extend(labels.numpy())
            y_scores.extend(probs.cpu().numpy())
            y_pred.extend((probs > 0.5).float().cpu().numpy())
    
    val_acc = accuracy_score(y_true, y_pred)
    val_eer = compute_eer(y_true, y_scores)
    elapsed = time.time() - start_time
    
    print(f"Epoch {epoch+1}/{EPOCHS} | Train Loss: {avg_train_loss:.4f} | "
          f"Val Acc: {val_acc*100:.2f}% | Val EER: {val_eer*100:.2f}% | "
          f"Time: {elapsed:.0f}s")
    
    # ---- CHECKPOINTING ----
    if val_eer < best_eer:
        best_eer = val_eer
        torch.save(model.state_dict(), "best_wav2vec_model.pth")
        print(f"  ✓ Saved best model (EER: {val_eer*100:.2f}%)")
        epochs_no_improve = 0
    else:
        epochs_no_improve += 1
        print(f"  No improvement ({epochs_no_improve}/{PATIENCE})")
        if epochs_no_improve >= PATIENCE:
            print("Early stopping triggered!")
            break

print(f"\nTraining complete! Best Val EER: {best_eer*100:.2f}%")
print("Download best_wav2vec_model.pth from the file sidebar.")


In [ ]:
# Download the trained model
from google.colab import files
files.download("best_wav2vec_model.pth")
